# PinholePose

A `PinholePose` couples a camera `Pose3` with a fixed calibration model. It is useful when the calibration is known ahead of time and fixed during optimization. This is different from `PinholeCamera`, which also optimizes the calibration itself. For simplicity, this guide demonstrates the `Cal3_S2` variant of `PinholePose` (`PinholePoseCal3_S2` in Python), though many other calibration models are supported:

- `Cal3_S2`
- `Cal3Bundler`
- `Cal3DS2`
- `Cal3Fisheye`
- `Cal3Unified`

## Understanding the Pinhole Pose
A *pinhole pose* packages the camera's pose in the world with a **fixed** calibration. Conceptually it acts like a `PinholeCamera` with constant intrinsics. When the intrinsics are known from a prior calibration step, optimizing only the pose can simplify many problems. Throughout this notebook we will refer to `PinholePoseCal3_S2`, which pairs a `Pose3` with a `Cal3_S2` instance.

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam/geometry/doc/PinholePose.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
%pip install --quiet gtsam-develop

Note: you may need to restart the kernel to use updated packages.


In [3]:
import gtsam
import numpy as np
from gtsam import PinholePoseCal3_S2, Pose2, Pose3, Point2, Point3, Cal3_S2

## Initialization

You can construct a `PinholePoseCal3_S2` with just a pose, in which case the calibration matrix is defaulted to the identity matrix $I_3$.

In [4]:
pose = Pose3()
camera0 = PinholePoseCal3_S2(pose)
print(camera0.pose())
print(camera0.calibration())

R: [
	1, 0, 0;
	0, 1, 0;
	0, 0, 1
]
t: 0 0 0

Cal3_S2[
	1, 0, 0;
	0, 1, 0;
	0, 0, 1
]



You can also construct a `PinholePoseCal3_S2` with a specific pose and calibration object.

In [5]:
cal = Cal3_S2(625, 625, 0, 320, 240)
camera1 = PinholePoseCal3_S2(pose, cal)
print(camera1.pose())
print(camera1.calibration())

R: [
	1, 0, 0;
	0, 1, 0;
	0, 0, 1
]
t: 0 0 0

Cal3_S2[
	625, 0, 320;
	0, 625, 240;
	0, 0, 1
]



### Named constructors

The class provides convenient constructors such as `Level` and `Lookat`. `Level` creates a camera at a given 2D pose and height, oriented level with the ground. `Lookat` creates a camera that looks from an eye point to a target with a chosen up direction.

In [15]:
Point3(1,1,1).dtype

dtype('float64')

In [21]:
K = Cal3_S2()
level_cam = PinholePoseCal3_S2.Level(Pose2(1.0, 2.0, 0.0), 1.5)
lookat_cam = PinholePoseCal3_S2.Lookat(Point3(0,0,0), Point3(1,0,0), Point3(0,0,1), K)
print(level_cam.pose())
print(lookat_cam.pose())

R: [
	0, 0, 1;
	-1, 0, 0;
	0, -1, 0
]
t:   1   2 1.5

R: [
	0, 0, 1;
	-1, 0, 0;
	0, -1, 0
]
t: 0 0 0



## Properties

Use `pose()` to access the camera pose and `calibration()` to access the fixed calibration.

In [22]:
print(camera1.pose())
print(camera1.calibration())

R: [
	1, 0, 0;
	0, 1, 0;
	0, 0, 1
]
t: 0 0 0

Cal3_S2[
	625, 0, 320;
	0, 625, 240;
	0, 0, 1
]



## Basic Operations

### `project()`
Use `project(point)` to project a 3D point into the image.


In [23]:
p = Point3(0.1, -0.1, 1.0)
uv = camera1.project(p)
print(uv)

[382.5 177.5]


### `backproject()`
Given an image point and a depth, `backproject` reconstructs the corresponding 3D point.

In [24]:
p2 = Point2(0, 0)
P = camera1.backproject(p2, 1.0)
print(P)

[-0.512 -0.384  1.   ]


### `range()`
The `range` method computes the distance from the camera to a landmark or another pose.

In [25]:
dist = camera1.range(Point3(1,2,3))
print(dist)

3.7416573867739413


### Manifold Operations

Like other geometric types, `PinholePose` is optimized on its manifold using `retract` and `localCoordinates`.

In [26]:
delta = np.array([0.1, 0.2, 0, 0.0, 0.0, 0.0])
cam2 = camera1.retract(delta)
print(camera1.localCoordinates(cam2))

[0.1 0.2 0.  0.  0.  0. ]


## Example: Using PinholePose in a Factor Graph

When camera intrinsics are known, `PinholePose` can serve as the variable representing a camera pose. The snippet below builds a minimal factor graph with one `GenericProjectionFactorCal3_S2`.

In [ ]:
graph = gtsam.NonlinearFactorGraph()
initial = gtsam.Values()

pose_key = gtsam.symbol('x', 0)
point_key = gtsam.symbol('p', 0)

measurement = Point2(320, 240)
noise = gtsam.noiseModel.Isotropic.Sigma(2, 1.0)
factor = gtsam.GenericProjectionFactorCal3_S2(measurement, noise, pose_key, point_key, cal)
graph.add(factor)

initial.insert(pose_key, camera1.pose())
initial.insert(point_key, Point3(1.0, 0.0, 5.0))

print(f'Graph has {graph.size()} factor(s) and {initial.size()} variable(s)')

## Additional Resources

- [Pinhole camera model (Wikipedia)](https://en.wikipedia.org/wiki/Pinhole_camera_model)
- [GTSAM Visual SLAM examples](https://github.com/borglab/gtsam/tree/develop/python/gtsam/examples)


## Source
- [PinholePose.h](https://github.com/borglab/gtsam/blob/develop/gtsam/geometry/PinholePose.h)